# Multiple Linear Regression & Logistic Regression

1. Multiple Linear Regression
    1. Multicollinearity
    2. Autocorrelation
    3. Heteroscedasticity
2. Logistic Regression
    1. Binary Logistic Regression
    2. Coefficients and Wald Statistics
    3. Model Fitting Information
    4. Likelihood Ratio Test
    5. Pseudo R-Square
    6. Classification
    7. Interaction Effects

In [ ]:
# Imports

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from scipy.stats import binom, chi2
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import roc_curve, roc_auc_score

pd.set_option("display.max_columns", None)

# 1. Multiple Linear Regression

In [ ]:
# Load Data
df = pd.read_excel(""
)

Research question: How do the number of salespersons, the price and the advertising budget influence quantity sold?

## Model 1: Salespersons, Price and Advertising

In [ ]:
# Define the independent variables used in the regression model
X_cols = ["salespersons", "price", "advertising"]

# Filter the dataset
sub = df[df["district"] == 1].sort_values("period").reset_index(drop=True)

# Select the independent variables and add a constant term to estimate the intercept
X = sm.add_constant(sub[X_cols])

# Define the dependent variable
y = sub["quantity"]

# Fit a multiple linear regression model using Ordinary Least Squares (OLS)
model = sm.OLS(y, X).fit()

# Output
print("\nMultiple Linear Regression - Model 1:")
print("-----------------------------")
print(model.summary())

## 1.1 Multicollinearity

- Variance inflation factor (VIF)
- VIF > 3: potential multicollinearity problem
- VIF > 10: serious multicollinearity problem
- Run additional regressions: In each regression, we use one of the original independent variables as a dependent variable and keep the others as independent variables.

In [ ]:
# Create a DataFrame to store the Variance Inflation Factor (VIF) for each independent variable
vif = pd.DataFrame({
    "Variable": X.columns,
     # Calculate the VIF
    "VIF": [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]
})

# Calculate the Tolerance
vif["Tolerance"] = 1 / vif["VIF"]

# Output
print("\nMulticollinearity Diagnostics - Model 1:")
print("-----------------------------")
print(vif)

## 1.2 Autocorrelation

Autocorrelation is present if residuals correlate with
each other.
- Most common in time series
analyses (unlike in our example)
- First-order autocorrelation: A high
𝜖𝑖−1 is associated with high/low 𝜖𝑖
- Indicated by Durbin-Watson statistic
$$
d_w = \frac{\sum_{i \in I'} (\epsilon_i - \epsilon_{i-1})^2}{\sum_{i \in I} \epsilon_i^2}
$$ , where 𝐼′ is the
index set of observations without
the first observation

**Interpretation of Durbin-Watson**:
- $DW= 0$: positive first-order autocorrelation – definitely problematic < 1
- $DW = 2$: no autocorrelation (Between 1.5 and 2.5)
- $DW = 4$: negative autocorrelation – definitely problematic > 3

We can solve autocorrelation by detecting the factors responsible for the temporal fluctuations.

In [ ]:
# Create a DataFrame containing the observed values, predicted values, and residuals for each period
results = pd.DataFrame({
    "period": sub["period"],
    "salespersons": sub["salespersons"],
    "price": sub["price"],
    "advertising": sub["advertising"],
    "y_i": y,
    "y_hat_i": model.fittedvalues,
    "epsilon_i": model.resid
})

# Calculate the first differences between consecutive residuals
results["epsilon_diff"] = results["epsilon_i"].diff()

# Square the differences between consecutive residuals
results["epsilon_diff_squared"] = results["epsilon_diff"] ** 2

# Output
display(results.round(3))

# Calculate the Durbin-Watson statistic
dw = durbin_watson(model.resid)

# Output
print(f"Durbin-Watson-Statistik: {dw:.4f}")

## 1.3 Heteroscedasticity

Heteroscedasticity is present if residuals do not have constant variance
→ Graphical inspection with scatterplot
Standardized residuals ~ Standardized predicted values

Tests for heteroscedasticity:
- Goldfeld-Quandt test
- Breusch-Pagan test
- White test

Solutions for heteroscedasticity:
- Transformation of variables
- Weighted (instead of ordinary) least squares
- Heteroscedastic-robust standard errors

**Interpretation of Breusch-Pagan**:
- $H_0$: No heteroscedasticity (homoscedasticity; constant error variance).
- $p > 0.05$: Fail to reject $H_0$, insufficient evidence of heteroscedasticity.
- $p \leq 0.05$: Reject $H_0$, evidence of heteroscedasticity.

In [ ]:
# Add a constant term to estimate the intercept
exog_fitted = sm.add_constant(
    pd.DataFrame({"fitted": model.fittedvalues})
)

# Perform the Breusch-Pagan test
bp_model1 = het_breuschpagan(
    model.resid,
    exog_fitted,
    robust=False
)

# Output
print("\nBreusch-Pagan Test - Model 1:")
print("-----------------------------")
print(f"LM Statistic (Chi²): {bp_model1[0]:.4f}")
print(f"LM p-value:          {bp_model1[1]:.4f}")
print(f"F Statistic:         {bp_model1[2]:.4f}")
print(f"F p-value:           {bp_model1[3]:.4f}")

In [ ]:
# Scatterplot

# Calculate internally studentized residuals
standardized_residuals = model.get_influence().resid_studentized_internal

# Standardize the predicted values
standardized_predicted = (
    (model.fittedvalues - model.fittedvalues.mean())
    / model.fittedvalues.std()
)

# Create a compact figure for residual diagnostics
plt.figure(figsize=(5, 3.2), dpi=120)

# Plot standardized residuals against standardized predicted values
plt.scatter(
    standardized_predicted,
    standardized_residuals,
    alpha=0.5,
    s=20
)

# Add reference lines at zero
plt.axhline(y=0, linestyle="--", linewidth=1)
plt.axvline(x=0, linestyle="--", linewidth=1)

# Set axis labels and title
plt.xlabel("Standardized Predicted Values")
plt.ylabel("Standardized Residuals")
plt.title("Residuals vs. Predicted Values", fontsize=11, pad=8)

# Add a subtle grid and adjust the layout
plt.grid(True, alpha=0.2, linestyle="--")
plt.tight_layout()

# Output
plt.show()

## Model 2: Salespersons, Price, Advertising and Number of Mailings

In [ ]:
# Select the independent variables for the multiple regression model
X_cols2 = sub[[
    "salespersons",
    "price",
    "advertising",
    "number of mailings"
]]

# Add a constant term to include the intercept in the regression model
X_model2 = sm.add_constant(X_cols2)

# Fit the multiple linear regression model using Ordinary Least Squares (OLS)
model2 = sm.OLS(y, X_model2).fit()

# Output
print("\nMultiple Linear Regression - Model 2:")
print("-----------------------------")
print(model2.summary())

## Multicollinearity Diagnostics – Model 2

In [ ]:
# Create a DataFrame to store the Variance Inflation Factor (VIF) for each independent variable
vif_model2 = pd.DataFrame({
    "Variable": X_model2.columns,
    # Calculate the VIF
    "VIF": [
        variance_inflation_factor(X_model2.values, i)
        for i in range(X_model2.shape[1])
    ]
})

# Calculate the Tolerance
vif_model2["Tolerance"] = 1 / vif_model2["VIF"]

# Output
print("\nMulticollinearity Diagnostics - Model 2:")
print("-----------------------------")
print(vif_model2)

# 2. Logistic Regression

In [ ]:
# Load Data

df2 = pd.read_csv(""
)

In [ ]:
# Prepare Data

df2["Pass_binary"] = (df2["Pass"] == "YesPass").astype(int)

df2["Channel"] = pd.Categorical(
    df2["Channel"],
    categories=["Park", "Email", "Mail"]
)

df2["Promo"] = pd.Categorical(
    df2["Promo"],
    categories=["NoBundle", "Bundle"]
)

Research question: How do the sales channel (email, postal mail or in-person at the park) and whether it was promoted in a bundle with another feature explain the purchase decision for a season pass in an amusement park?

## 2.1 Binary Logistic Regression

**Exp(B)**: The relative change in the odds ratio
  $\left(\frac{P(\text{YesPass})}{P(\text{NoPass})}\right)$
  associated with a one-unit increase in the independent variable

In [ ]:
# Fit a binary logistic regression model to predict pass outcomes
# Use Channel and Promo as categorical independent variables
logit_model = smf.logit(
    "Pass_binary ~ C(Channel) + C(Promo)",
    data=df2
).fit()

# Output
print("\nBinary Logistic Regression:")
print("-----------------------------")
print(logit_model.summary())

#### Odds Ratio

The odds of an event are the probability that it occurs divided by the probability that it does not occur $$
\text{Odds} = \frac{p}{1 - p}
$$ The odds ratio compares the odds of success between two groups, or for a one-unit increase in a predictor.

- $OR = 1$: no effect
- $OR > 1$: higher odds of success
- $OR < 1$: lower odds of success (e.g. $OR = 0.57$ means 43% lower odds)

**Note:** Changes in the *odds* are not the same as changes in the probability of success. For example, a 98% reduction in odds does not mean the probability of success falls by 98 percentage points.

In [ ]:
# Calculate the 95% confidence intervals for the model coefficients
conf = logit_model.conf_int()

# Create a DataFrame containing odds ratios, confidence intervals, and p-values
odds_df = pd.DataFrame({
    "Odds Ratio": np.exp(logit_model.params),
    "CI 2.5%": np.exp(conf[0]),
    "CI 97.5%": np.exp(conf[1]),
    "p-value": logit_model.pvalues
})

# Calculate the percentage change in odds relative to an odds ratio of 1
odds_df["Change in odds (%)"] = (odds_df["Odds Ratio"] - 1) * 100

# Convert the confidence interval bounds into percentage changes in odds
odds_df["CI 2.5% (%)"] = (odds_df["CI 2.5%"] - 1) * 100
odds_df["CI 97.5% (%)"] = (odds_df["CI 97.5%"] - 1) * 100

# Output (without Intercept)
print("Odds Ratios and % Change in Odds:")
print("-----------------------------")
print(odds_df.drop("Intercept").round(3))

## 2.2 Coefficients and Wald Statistics

- **z-value**: The z-value is the estimated coefficient divided by its standard error ($z = b / SE$) and indicates how many standard errors the coefficient is away from zero
- **Wald-Statistic**: The Wald statistic is the squared z-value ($W = z^2$) and tests the same null hypothesis ($H_0$: $b = 0$) against a $\chi^2$ distribution with 1 degree of freedom
- Both lead to the same conclusion and p-value

In [ ]:
# Extract the coefficient table from the logistic regression summary
coef_table = logit_model.summary2().tables[1].copy()

# Calculate the Wald chi-square statistic by squaring the z-statistic
coef_table.insert(3, "Wald", coef_table["z"] ** 2)

# Calculate the odds ratios by exponentiating the estimated coefficients
coef_table.insert(4, "Exp(B)", np.exp(coef_table["Coef."]))

# Output
print("\nCoefficients:")
print("-----------------------------")
print(coef_table.round(3))

## 2.3 Model Fitting Information

In [ ]:
# Get the number of observations in the fitted logistic regression model
n = logit_model.nobs

# Calculate the number of parameters in the full model, including the intercept
k_full = logit_model.df_model + 1

# Set the number of parameters in the null model, which contains only an intercept
k_null = 1

# Aggregate the binary outcomes by Channel and Promo
agg = (
    df2.groupby(["Channel", "Promo"], observed=True)["Pass_binary"]
    .agg(y="sum", n="count")
    .reset_index()
)

# Predict the probability of a successful outcome for each Channel-Promo combination
agg["p_final"] = logit_model.predict(
    agg[["Channel", "Promo"]]
)

# Calculate the overall success rate to use as the predicted probability for the null model
p_null = df2["Pass_binary"].mean()

# Calculate the log-likelihood of the full model using the aggregated binomial outcomes
# Sum the log-probabilities across all Channel-Promo groups
ll_full = binom.logpmf(
    agg["y"], agg["n"], agg["p_final"]
).sum()

# Calculate the log-likelihood of the null model using the overall success rate
# Sum the log-probabilities across all Channel-Promo groups
ll_null = binom.logpmf(
    agg["y"], agg["n"], p_null
).sum()

### Model Fitting Criteria

**Information Criteria AIC and BIC**
- Used to compare multiple models
- Take into account number of coefficients (safeguard against overfitting)
- Smaller is better

In [ ]:
# Create a DataFrame to compare the model fit criteria for the null and final models
fit_criteria = pd.DataFrame(
    {
        # Calculate the Akaike Information Criterion (AIC) for both models
        "AIC": [
            -2 * ll_null + 2 * k_null,
            -2 * ll_full + 2 * k_full
        ],
         # Calculate the Bayesian Information Criterion (BIC) for both models
        "BIC": [
            -2 * ll_null + np.log(n) * k_null,
            -2 * ll_full + np.log(n) * k_full
        ],
        # Calculate minus twice the log-likelihood for both models
        "-2 Log Likelihood": [
            -2 * ll_null,
            -2 * ll_full
        ],
    },
    index=["Intercept Only", "Final"]
)

# Output
print("\nModel Fitting Criteria:")
print("-----------------------------")
print(fit_criteria.round(3))

## 2.4 Likelihood Ratio Test

The Likelihood Ratio Test (LRT) compares nested models to determine whether the more complex model provides a significantly better fit to the data than the simpler model.

In [ ]:
# Create a DataFrame summarizing the likelihood ratio test comparing the final model against the intercept-only (null) model
lr_test = pd.DataFrame(
    {
        "Chi-Square": [logit_model.llr],
        "df": [int(logit_model.df_model)],
        "Sig.": [logit_model.llr_pvalue]
    },
    index=["Final vs. Intercept Only"]
)

# Output
print("\nLikelihood Ratio Test:")
print("-----------------------------")
print(lr_test.round(3))

## 2.5 Pseudo R-Square

**McFadden’s 𝑅2**
- Interpretation similar to 𝑅2 in linear regression.
- Often between 0,2 and 0,4 is often said to be indicative of good model fit

In [ ]:
# Retrieve the number of observations and the log-likelihoods of the fitted and null models
n = logit_model.nobs
ll_full = logit_model.llf
ll_null = logit_model.llnull

# Calculate the Cox and Snell pseudo-R-squared
r2_cox_snell = 1 - np.exp((2 / n) * (ll_null - ll_full))

# Calculate the Nagelkerke pseudo-R-squared
r2_nagelkerke = r2_cox_snell / (
    1 - np.exp((2 / n) * ll_null)
)

# Calculate the McFadden pseudo-R-squared
r2_mcfadden = 1 - (ll_full / ll_null)

# Create a DataFrame summarizing the three pseudo-R-squared measures
pseudo_r2 = pd.DataFrame(
    {"Pseudo R-Square": [
        r2_cox_snell,
        r2_nagelkerke,
        r2_mcfadden
    ]},
    index=["Cox and Snell", "Nagelkerke", "McFadden"]
)

# Output
print("\nPseudo R-Square:")
print("-----------------------------")
print(pseudo_r2.round(3))

## 2.6 Classification


- **Sensitivity (True Positive Rate):** The proportion of observed positive cases that the model correctly predicts as positive.

  $$
  \text{Sensitivity} = \frac{TP}{TP + FN}
  $$


- **Specificity (True Negative Rate):** The proportion of observed negative cases that the model correctly predicts as negative.

  $$
  \text{Specificity} = \frac{TN}{TN + FP}
  $$


- **Precision (Positive Predictive Value):** The proportion of predicted positive cases that are actually positive.

  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$


- **Accuracy:** The proportion of all cases that are classified correctly.

  $$
  \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
  $$

In [ ]:
# Extract the observed binary outcomes and predicted probabilities from the fitted logistic regression model
y_obs = logit_model.model.endog
p_hat = logit_model.predict()

# Classify observations using a probability threshold of 0.5
y_pred = (p_hat >= 0.5).astype(int)

# Map binary outcomes to descriptive class labels
labels = {0: "NoPass", 1: "YesPass"}
order = ["NoPass", "YesPass"]

# Create a confusion matrix comparing observed and predicted classes
cm = pd.crosstab(
    pd.Series(y_obs).map(labels).rename("Observed"),
    pd.Series(y_pred).map(labels).rename("Predicted")
).reindex(
    index=order,
    columns=order,
    fill_value=0
)

# Extract the four cells of the confusion matrix (positive class = YesPass)
TN = cm.loc["NoPass", "NoPass"]
FP = cm.loc["NoPass", "YesPass"]
FN = cm.loc["YesPass", "NoPass"]
TP = cm.loc["YesPass", "YesPass"]

# Calculate classification metrics based on the confusion matrix
accuracy = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)  # True Positive Rate (Recall)
specificity = TN / (TN + FP)  # True Negative Rate
precision = TP / (TP + FP)  # Positive Predictive Value

# Copy the confusion matrix to include classification performance metrics
class_table = cm.copy()

# Percentage correctly classified within each actual class
# (NoPass row = Specificity, YesPass row = Sensitivity)
class_table["Percent Correct"] = [
    100 * cm.loc[cat, cat] / cm.loc[cat].sum()
    for cat in order
]

# Add an overall percentage row showing the distribution of predictions across classes and the overall accuracy
class_table.loc["Overall Percentage"] = [
    100 * cm["NoPass"].sum() / cm.values.sum(),
    100 * cm["YesPass"].sum() / cm.values.sum(),
    100 * accuracy
]

# Output
print("\nClassification Table:")
print("-----------------------------")
print(class_table.round(1))

print(f"\nAccuracy:                     {accuracy:.3f}")
print(f"Sensitivity (Recall, TPR):    {sensitivity:.3f}")
print(f"Specificity (TNR):            {specificity:.3f}")
print(f"Precision (PPV):              {precision:.3f}")

# Calculate the baseline accuracy achieved by always predicting the majority class
baseline = cm.sum(axis=1).max() / cm.values.sum()

# Output
print(f"Baseline accuracy:            {baseline:.3f}")

In [ ]:
# Calculate the false positive rate (FPR), true positive rate (TPR), and classification thresholds for different decision cutoffs
fpr, tpr, thresholds = roc_curve(y_obs, p_hat)

# Calculate the Area Under the ROC Curve (AUC)
auc = roc_auc_score(y_obs, p_hat)

# Create a compact figure for evaluating classification performance
plt.figure(figsize=(4.5, 3), dpi=120)

# Plot the ROC curve to visualize the trade-off between sensitivity and the false positive rate across classification thresholds
plt.plot(
    fpr,
    tpr,
    linewidth=1.8,
    label=f"AUC = {auc:.3f}"
)

# Add the diagonal reference line representing random classification
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1
)

# Label the axes and add a descriptive title
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve", fontsize=11, pad=8)

# Display the AUC value in the legend
plt.legend(fontsize=9)

# Add a subtle grid and adjust the layout
plt.grid(True, alpha=0.2, linestyle="--")
plt.tight_layout()

# Output
plt.show()
print(f"\nAUC: {auc:.3f}")

## 2.7 Interaction Model: Channel × Promo

Maybe there is a different effect of Promo depending on Channel: Interaction effect

In [ ]:
# Fit a binary logistic regression model including the interaction between Channel and Promo
logit_model_int = smf.logit(
    "Pass_binary ~ C(Channel) * C(Promo)",
    data=df2
).fit()

# Output
print("\nLogistic Regression Results (with interaction):")
print("-----------------------------")
print(logit_model_int.summary())

### Model Coefficients

In [ ]:
# Calculate the confidence intervals for the estimated logistic regression coefficients
confidence_intervals = logit_model_int.conf_int()

# Create a DataFrame summarizing the coefficients and their statistical significance and the corresponding odds ratios with 95% confidence intervals
coefficients = pd.DataFrame({
    "Coefficient": logit_model_int.params,
    "Std Error": logit_model_int.bse,
    "Wald": (logit_model_int.params / logit_model_int.bse) ** 2,
    "df": 1,
    "p-value": logit_model_int.pvalues,
    "Odds Ratio": np.exp(logit_model_int.params),
    "CI Lower": np.exp(confidence_intervals[0]),
    "CI Upper": np.exp(confidence_intervals[1])
})

# Calculate the percentage change in odds: (OR - 1) * 100
coefficients["Change in odds (%)"] = (coefficients["Odds Ratio"] - 1) * 100
coefficients["CI Lower (%)"] = (coefficients["CI Lower"] - 1) * 100
coefficients["CI Upper (%)"] = (coefficients["CI Upper"] - 1) * 100

# Output
print("\nCoefficient Table:")
print("-----------------------------")
print(coefficients.round(3))

### Likelihood Ratio Test

In [ ]:
# Calculate the likelihood ratio (LR) chi-square statistic to compare the interaction model with the main-effects-only model
lr_chi2_int = 2 * (logit_model_int.llf - logit_model.llf)

# Calculate the difference in degrees of freedom between the two models
lr_df_int = logit_model_int.df_model - logit_model.df_model

# Calculate the p-value from the chi-square distribution
lr_p_int = chi2.sf(lr_chi2_int, lr_df_int)

# Create a DataFrame summarizing the likelihood ratio test results comparing the interaction model with the main-effects model
lr_test_int = pd.DataFrame(
    {
        "Chi-Square": [lr_chi2_int],
        "df": [int(lr_df_int)],
        "p-value": [lr_p_int]
    },
    index=["Interaction vs. Main Effects"]
)

# Output
print("\nLikelihood Ratio Test:")
print("-----------------------------")
print(lr_test_int.round(3))

## Promo Effect Within Each Channel

In [ ]:
# Get the names of the estimated coefficients from the interaction model
names = logit_model_int.params.index

# Define the coefficient representing the effect of the Bundle promotion
promo = "C(Promo)[T.Bundle]"

# Create a contrast vector to test a linear combination of model coefficients
def contrast(*terms):
    c = np.zeros(len(names))
    for term in terms:
        c[names.get_loc(term)] = 1
    return c

# Define the promotion effect within each channel
promo_effects = {
    "Park": contrast(promo),
    "Email": contrast(promo, f"C(Channel)[T.Email]:{promo}"),
    "Mail": contrast(promo, f"C(Channel)[T.Mail]:{promo}")
}

# Initialize a list to store the results for each channel
rows = []

# Estimate and test the promotion effect separately for each channel
for channel, contrast_vector in promo_effects.items():
    # Perform a hypothesis test for the specified linear combination of coefficients
    result = logit_model_int.t_test(contrast_vector)

    # Extract the estimated effect, standard error, and p-value
    coefficient = float(np.squeeze(result.effect))
    std_error = float(np.squeeze(result.sd))
    p_value = float(np.squeeze(result.pvalue))

    # Extract the lower and upper confidence limits on the coefficient scale
    lower, upper = np.squeeze(result.conf_int())

 # Store the effect estimate, its statistical significance, and the odds ratio
    rows.append({
        "Channel": channel,
        "Coefficient": coefficient,
        "Std Error": std_error,
        "p-value": p_value,
        "Odds Ratio": np.exp(coefficient),
        "CI Lower": np.exp(lower),
        "CI Upper": np.exp(upper)
    })

# Create a DataFrame with one row per channel
promo_effects = pd.DataFrame(rows).set_index("Channel")

# Output
print("\nPromo Effect within Channel:")
print("-----------------------------")
print(promo_effects.round(3))